In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import torch
import torch.nn as nn
import pandas as pd
import numpy as np

from src.utils.config import load_config
from src.data.load_data import load_exchange_data
from src.data.window import create_windows
from src.data.dataloader import create_dataloaders
from src.evaluation.metrics import smape
from src.evaluation.arima import arima_multivariate
from src.training.train import train_model
from src.utils.positional import PositionalEncoding
from src.experiments.run_experiment import run_experiment

/Users/akarn/Documents/Exchange_curr_pred/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)


Using device: mps


In [3]:
train_df, val_df, test_df = load_exchange_data(univariate=False)

print(train_df.shape, val_df.shape, test_df.shape)


(5311, 8) (759, 8) (1518, 8)


In [4]:
pred_len = 199

arima_val_smape = arima_multivariate(
    train_df,
    val_df,
    order=(1,1,1)
)

print("ARIMA Validation sMAPE:", arima_val_smape)


/Users/akarn/Documents/Exchange_curr_pred/.venv/lib/python3.14/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


ARIMA Validation sMAPE: 0.6244915266626219


In [5]:
train_plus_val = np.vstack([train_df, val_df])
test_array = np.array(test_df)

arima_test_smape = arima_multivariate(
    train_plus_val,
    test_array,
    order=(1,1,1)
)


print("ARIMA Test sMAPE:", arima_test_smape)


/Users/akarn/Documents/Exchange_curr_pred/.venv/lib/python3.14/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


ARIMA Test sMAPE: 0.7054108053298362


In [6]:
results_table=[]
config = load_config(
    "src/configs/base.yaml",
    overrides={
        "model_type": "transformer",
        "seq_len": 192,
        "pred_len": 199,
        "d_model": 64,
        "n_heads": 4,
        "enc_layers": 2,
        "dropout": 0.1,
        "epochs": 20,
        "residual": True
    }
)

res = run_experiment(config)
results_table.append({
            "name": config["save_path"].rsplit(sep='/')[7].split(".")[0],
            **res

            })
results_table

Epoch 1 | Train Loss 0.3408 | Val sMAPE 0.1834
Epoch 2 | Train Loss 0.2704 | Val sMAPE 0.2991
Epoch 3 | Train Loss 0.2621 | Val sMAPE 0.2951
Epoch 4 | Train Loss 0.2471 | Val sMAPE 0.2220
Epoch 5 | Train Loss 0.2214 | Val sMAPE 0.2988
Epoch 6 | Train Loss 0.2276 | Val sMAPE 0.3211
Early stopping triggered.


[{'name': 'transformer_s192_p199_h4_k23_d0.1',
  'model_type': 'transformer',
  'seq_len': 192,
  'pred_len': 199,
  'n_heads': 4,
  'dropout': 0.1,
  'd_model': 64,
  'enc_layers': 2,
  'model_smape': 0.18344337679445744,
  'naive_smape': 1.6775708198547363,
  'beats_naive': True}]